In [16]:
import sys
import os

sys.path.append(os.path.abspath(".."))
from src.preprocess import process_pipeline

df = process_pipeline("../data/processed/transformed_spam.csv")
df.head()


,label,message,transformed_message
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,0,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entri wkli comp win fa cup final tkt st m...
3,0,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah dont think goe usf live around though


In [17]:
print(df['transformed_message'].isnull().sum())
print(df['transformed_message'].dtype)
print(df['transformed_message'].head())

6
str
0    go jurong point crazi avail bugi n great world...
1                                ok lar joke wif u oni
2    free entri wkli comp win fa cup final tkt st m...
3                  u dun say earli hor u c alreadi say
4            nah dont think goe usf live around though
Name: transformed_message, dtype: str


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

# FIX: Handle null values
df['transformed_message'] = df['transformed_message'].fillna('')

X = vectorizer.fit_transform(df['transformed_message'])
y = df['label']

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [20]:
from sklearn.naive_bayes import MultinomialNB
model=MultinomialNB()
model.fit(X_train,y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [21]:
#prediction 
y_pred=model.predict(X_test)

In [24]:
#Evaluate Model
from sklearn.metrics import accuracy_score,precision_score,confusion_matrix
print("Accuracy:",accuracy_score(y_test,y_pred))
print("Precision:",precision_score(y_test,y_pred))
print("confusion matrix:",confusion_matrix(y_test,y_pred))

Accuracy: 0.9650224215246637
Precision: 1.0
confusion matrix: [[965   0]
 [ 39 111]]


In [25]:
from sklearn.metrics import recall_score, f1_score

print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

Recall: 0.74
F1 Score: 0.8505747126436781


In [26]:
#Logistic Regression 
from sklearn.linear_model import LogisticRegression

lr=LogisticRegression()
lr.fit(X_train,y_train)
y_pred_lr=lr.predict(X_test)
print("Accuracy:",accuracy_score(y_test,y_pred_lr))
print("precision:",precision_score(y_test,y_pred_lr))


Accuracy: 0.9479820627802691
precision: 0.96


I started with a labeled dataset containing spam and non-spam messages. I performed data cleaning and preprocessing steps like lowercasing, removing punctuation and stopwords, and text normalization.
Then I converted the text into numerical features using TF-IDF vectorization, which captures the importance of words across the dataset.
For model training, I initially used Multinomial Naive Bayes as a baseline, since it works well for text classification and high-dimensional sparse data.

In [30]:
#SVM

from sklearn.svm import SVC

svm_model = SVC(kernel='linear')
svm_model.fit(X_train, y_train)

y_pred = svm_model.predict(X_test)


In [31]:

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.9766816143497757
Precision: 0.9769230769230769
Recall: 0.8466666666666667
F1 Score: 0.9071428571428571
Confusion Matrix:
 [[962   3]
 [ 23 127]]


compared Naive Bayes, Logistic Regression, and SVM. Naive Bayes gave the highest precision, SVM provided a better balance between precision and recall, and Logistic Regression performed moderately. Based on business requirements, I would choose the model accordingly

In [32]:
import pickle
pickle.dump(model,open("../models/naive_bayes.pkl","wb"))
pickle.dump(lr,open("../models/logistic_regression.pkl","wb"))
pickle.dump(svm_model,open("../models/svm_model.pkl","wb"))